# State

In [1]:
from typing import TypedDict

class State(TypedDict):
    operation: str
    processed_value: int


In [ ]:
from pydantic import BaseModel, Field
import uuid

class States(BaseModel):
    messages: list
    flag : bool = Field(default=False)
    userId: str = Field(default_factory=lambda: str(uuid.uuid4()))

# state = States()
# print(state.userId)

# Graph

In [2]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(State)

# Node

In [3]:
def addition_node(state: State):
    value = state.get('processed_value')
    print("--- add node executed ---")
    return {
        'processed_value' : value+2
    }

def sub_node(state: State):
    value = state.get('processed_value')
    print("--- subraction node executed ---")
    return {'processed_value': value-1}



In [4]:
graph.add_node("addNode",addition_node)
graph.add_node("subNode",sub_node)

# Edge

In [16]:
graph.add_edge(START, "addNode")
graph.add_edge("addNode", "subNode")
graph.add_edge("subNode", END)

In [18]:
agent = graph.compile()
response = agent.invoke({'processed_value':3})
print(response['processed_value'])

--- add node executed ---
--- subraction node executed ---
4


In [19]:
from PIL import Image
import io

img_bytes = agent.get_graph().draw_mermaid_png()
image = Image.open(io.BytesIO(img_bytes))
image.show()

# Conditional Edges

In [5]:
def router_function(state: State):
    # Note router function can only have access to read the state value not to write.
    operation = state.get('operation')
    if operation == 'add': return "add"
    elif operation == 'sub': return "sub"

graph.add_conditional_edges(
    START,
    router_function,
    {
        'add': 'addNode', # key: router function return data, Value: name of the node to be excuted.
        'sub':'subNode'
    }
)
graph.add_edge('addNode', END)
graph.add_edge('subNode', END)

agent = graph.compile()

In [8]:
response = agent.invoke({'processed_value': 4, 'operation':'sub'})
print(response['processed_value'])

--- subraction node executed ---
3


In [6]:
from PIL import Image
import io

img_bytes = agent.get_graph().draw_mermaid_png()
image = Image.open(io.BytesIO(img_bytes))
image.show()

# Custom Reducer

In [8]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END

# --- 1. Define the Custom Reducer Functions ---

def keep_unique_items(old_list: list, new_list: list) -> list:
    """Reducer: Appends new items to the list only if they aren't already in it."""
    if old_list is None: old_list = []
    if new_list is None: new_list = []
    
    # Combine lists but filter out duplicates
    return old_list + [item for item in new_list if item not in old_list]

def keep_highest_score(old_score: int, new_score: int) -> int:
    """Reducer: Only updates the score if the new score is higher."""
    if old_score is None: return new_score
    return max(old_score, new_score)

# --- 2. Apply Reducers to the State ---

class GameState(TypedDict):
    # Apply our custom functions using Annotated
    inventory: Annotated[list, keep_unique_items]
    top_score: Annotated[int, keep_highest_score]


# --- 3. Define the Nodes ---

def level_one(state: GameState):
    print("Playing Level 1...")
    # Returning a score of 50, and finding a sword and shield
    return {
        "inventory": ["sword", "shield"], 
        "top_score": 50
    }

def level_two(state: GameState):
    print("Playing Level 2...")
    # Returning a lower score (30). The reducer should ignore this.
    # Finding a potion, and another shield. The reducer should ignore the second shield.
    return {
        "inventory": ["potion", "shield"], 
        "top_score": 30
    }

# --- 4. Build and Run the Graph ---

builder = StateGraph(GameState)
builder.add_sequence([level_one, level_two])
builder.add_edge(START, "level_one")
builder.add_edge("level_two", END)

graph = builder.compile()

if __name__ == "__main__":
    # Initialize the state
    initial_state = {"inventory": [], "top_score": 0}
    
    # Run the graph
    result = graph.invoke(initial_state)
    
    print("\n--- Final State ---")
    print(f"Inventory: {result['inventory']}") 
    print(f"Top Score: {result['top_score']}")

Playing Level 1...
Playing Level 2...

--- Final State ---
Inventory: ['sword', 'shield', 'potion']
Top Score: 50
